# Evaluation, Baselines, Traditional ML:

We'll write some simple models to predict the price of a product.

We'll use an approach to evaluate the performance of the model.

And we'll test some Baseline Models using Traditional machine learning.

In [1]:
# Imports:
import random
from colorsys import yiq_to_rgb

import pandas as pd
import numpy as np
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.ensemble import RandomForestRegressor
from torch.profiler import python_tracer

from pricer.evaluator import evaluate
from pricer.items import Item

## Loading Dataset:

In [2]:
username = 'ed-donner'
dataset = f"{username}/items_lite"
train, val, test = Item.from_hub(dataset)
print(f"Loaded {len(train)} training items, {len(val)} validation items, {len(test)} test items.")

Loaded 20000 training items, 1000 validation items, 1000 test items.


## Model 1: Random Price Guesser

In [3]:
def random_price_guesser(item):
    return random.randrange(start= 1, stop= 1000)

In [8]:
# Evaluating Random Price Guesser Model using 200 Items from Test Data:

random.seed(42)
evaluate(function= random_price_guesser, data= test, size= 200, workers= 5)

  0%|          | 0/200 [00:00<?, ?it/s]

$436 $1 $29 $690 $252 $21 $85 $72 $719 $225 $20 $380 $894 $505 $11 $572 $354 $17 $179 $23 $90 $115 $433 $442 $304 $122 $291 $714 $567 $639 $539 $370 $66 $380 $489 $534 $769 $835 $207 $740 $626 $84 $680 $178 $129 $260 $142 $189 $836 $580 $310 $25 $380 $270 $47 $234 $861 $313 $417 $259 $591 $33 $657 $361 $79 $38 $757 $500 $263 $5 $534 $284 $570 $625 $584 $871 $759 $361 $575 $178 $602 $60 $17 $579 $207 $732 $115 $224 $756 $193 $866 $9 $370 $250 $456 $423 $821 $217 $103 $195 $264 $98 $650 $135 $470 $842 $675 $264 $43 $325 $591 $138 $516 $619 $56 $2 $449 $369 $221 $845 $640 $616 $501 $59 $502 $273 $844 $688 $616 $81 $164 $705 $52 $795 $259 $396 $70 $2 $89 $798 $902 $331 $818 $716 $129 $186 $627 $2 $141 $873 $918 $553 $423 $7 $253 $136 $204 $707 $255 $502 $19 $739 $557 $426 $80 $575 $39 $321 $185 $132 $497 $484 $326 $751 $53 $733 $85 $64 $549 $71 $158 $672 $133 $362 $16 $373 $258 $544 $420 $515 $223 $944 $532 $743 $866 $68 $527 $459 $67 $673 

## Model 2: Constant Price Guesser

In [9]:
# Finding Average Price from Training Data:

training_prices = [item.price for item in train]
training_average = sum(training_prices) / len(training_prices)
print(training_average)

140.347293000021


In [10]:
# Constant Price Guesser:
def constant_price_guesser(item):
    return training_average

In [11]:
# Evaluating Constant Price Guesser:
evaluate(function= constant_price_guesser, data= test, size= 200, workers= 5)

  0%|          | 0/200 [00:00<?, ?it/s]

$79 $24 $85 $70 $110 $90 $4 $75 $105 $190 $573 $239 $120 $86 $61 $107 $61 $90 $70 $21 $6 $16 $55 $35 $192 $312 $355 $120 $42 $60 $120 $80 $20 $60 $25 $679 $81 $84 $73 $102 $59 $60 $105 $115 $80 $115 $122 $109 $5 $62 $105 $10 $335 $21 $87 $6 $133 $100 $62 $128 $96 $62 $49 $30 $488 $50 $100 $305 $15 $64 $109 $123 $140 $121 $90 $104 $16 $130 $123 $122 $20 $128 $110 $41 $113 $80 $42 $166 $20 $95 $118 $45 $120 $105 $132 $88 $106 $17 $130 $435 $40 $23 $103 $1 $109 $22 $115 $260 $109 $159 $80 $174 $109 $12 $55 $30 $116 $120 $84 $37 $124 $51 $70 $26 $60 $80 $120 $41 $39 $1 $69 $3 $55 $110 $75 $125 $65 $70 $12 $2 $76 $110 $60 $121 $54 $108 $95 $370 $125 $107 $121 $34 $93 $0 $121 $139 $91 $84 $179 $90 $149 $114 $98 $127 $700 $117 $307 $90 $100 $130 $115 $118 $280 $118 $39 $9 $112 $47 $46 $47 $514 $115 $160 $108 $90 $118 $7 $73 $80 $114 $105 $89 $105 $2 $40 $60 $30 $140 $89 $114 

## Model 3: Linear Regression

In [17]:
# Making Features from Data:

def get_features(item):
    return {
        'Weight': item.weight,
        'Weight_Unknown': 1 if item.weight == 0 else 0,
        'Text_Length': len(item.summary)
    }

In [18]:
# Making Dataframe from Features:
def list_to_dataframe(items):
    features = [get_features(item) for item in items]
    df = pd.DataFrame(features)
    df['Price'] = [item.price for item in items]
    return df

In [19]:
# Making Dataframes from Training and Test Data:
train_df = list_to_dataframe(train)
test_df = list_to_dataframe(test)

In [20]:
# Linear Regression:
np.random.seed(42)

# Separating Features and Target:
features = ['Weight', 'Weight_Unknown', 'Text_Length']

# Training And Test Data:
X_train = train_df[features]
y_train = train_df['Price']
X_test = test_df[features]
y_test = test_df['Price']

# Fitting a Linear Regression Model:
model = LinearRegression()
model.fit(X_train, y_train)

# Feature Co-Efficients:
for feature, coef in zip(features, model.coef_):
    print(f"{feature}: {coef}")
print(f"Intercept: {model.intercept_}")

# Predicting on test Set and Evaluation:
y_pred = model.predict(X_test)
mse = mean_squared_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

print(f"MSE: {mse}")
print(f"R2: {r2}")

Weight: 3.5687444142553026
Weight_Unknown: 20.90175988023756
Text_Length: 0.20343123739151991
Intercept: 40.91238780791434
MSE: 20096.925335647466
R2: 0.1609102308229894


In [21]:
# Defining Function for Linear Regression:
def linear_regression_pricer(item):
    features = get_features(item)
    features_df = pd.DataFrame([features])
    return model.predict(features_df)[0]

In [22]:
# Evaluating Linear Regression:
evaluate(function= linear_regression_pricer, data= test, size= 200, workers= 5)

  0%|          | 0/200 [00:00<?, ?it/s]

$79 $51 $62 $66 $71 $50 $9 $50 $82 $191 $574 $233 $113 $58 $28 $97 $50 $62 $39 $14 $18 $14 $75 $11 $193 $318 $359 $91 $35 $36 $104 $78 $27 $16 $79 $676 $65 $66 $70 $67 $64 $36 $61 $66 $104 $93 $104 $83 $25 $60 $84 $5 $347 $2 $79 $20 $110 $83 $23 $123 $111 $49 $19 $3 $241 $17 $82 $261 $6 $68 $85 $105 $155 $89 $91 $80 $3 $98 $99 $88 $72 $86 $86 $8 $84 $43 $81 $176 $62 $60 $88 $42 $80 $71 $91 $117 $78 $8 $146 $418 $33 $7 $106 $21 $122 $1 $83 $298 $104 $198 $62 $154 $106 $59 $71 $53 $103 $94 $82 $14 $99 $43 $51 $51 $127 $47 $86 $102 $72 $16 $53 $9 $47 $72 $45 $84 $81 $40 $9 $11 $39 $132 $55 $85 $3 $71 $71 $364 $5 $72 $93 $26 $53 $21 $90 $119 $103 $40 $43 $79 $160 $92 $67 $116 $725 $100 $279 $74 $84 $101 $86 $100 $244 $78 $139 $58 $95 $7 $50 $38 $289 $117 $38 $94 $75 $96 $6 $77 $57 $79 $79 $78 $108 $21 $14 $97 $50 $109 $90 $114 